# RAG Assignment: Baseline Retrieval vs Cross-Encoder Re-Ranking

This notebook implements the assignment one stage at a time over the EU Artificial Intelligence Act.

## Step 1: PDF Extraction and Light Cleaning

Goal for this step:

- Read the full PDF.
- Extract text page by page.
- Preserve page numbers.
- Keep both `raw_text` and `clean_text`.
- Save the extracted pages to `outputs/pages.csv`.

The cleaning in this step is deliberately conservative. We clean whitespace and obvious repeated footer noise, but we do not rewrite the legal text.

In [ ]:
from pathlib import Path
import os
import re
import time

import numpy as np
import pandas as pd
import pymupdf
from sentence_transformers import CrossEncoder, SentenceTransformer

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "EU_Artificial_Intelligence_Act.pdf").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

PDF_PATH = PROJECT_ROOT / "data" / "EU_Artificial_Intelligence_Act.pdf"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
PAGES_CSV = OUTPUT_DIR / "pages.csv"
CHUNKS_CSV = OUTPUT_DIR / "chunks.csv"
EMBEDDINGS_PATH = OUTPUT_DIR / "chunk_embeddings.npy"
BASELINE_RESULTS_CSV = OUTPUT_DIR / "baseline_retrieval_results.csv"
RERANKED_RESULTS_CSV = OUTPUT_DIR / "reranked_results.csv"
LATENCY_RESULTS_CSV = OUTPUT_DIR / "latency_results.csv"
LLM_PROMPTS_CSV = OUTPUT_DIR / "llm_prompts.csv"
LLM_ANSWERS_CSV = OUTPUT_DIR / "llm_answers.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PDF_PATH

### Cleaning Policy

For this first step, cleaning is intentionally minimal:

- convert line breaks within a page into spaces,
- collapse repeated whitespace,
- remove obvious Official Journal footer fragments,
- preserve article numbers, annex labels, paragraph numbers, citations, and list structure as much as possible.

We are not using broad rules to join split words such as `ar tif icial` yet. If those artifacts hurt retrieval, we can add a small explicit replacement dictionary later.

In [ ]:
def is_header_footer_line(line: str) -> bool:
    """Return True for repeated Official Journal header/footer lines."""
    normalized = re.sub(r"\s+", " ", line).strip()
    lowered = normalized.lower()

    exact_noise = {
        "en",
        "official journal of the european union",
        "official journal",
        "offi cial jour nal of the european union",
        "offi cial jour nal",
        "of the european union",
        "l series",
        "l ser ies",
        "2024/1689",
        "12.7.2024",
    }
    if lowered in exact_noise:
        return True

    if re.fullmatch(r"OJ\s+L,\s*12\.7\.2024", normalized):
        return True
    if re.fullmatch(r"\d+\s*/\s*144", normalized):
        return True
    if "ELI: http://data.europa.eu/eli/reg/2024/1689/oj" in normalized:
        return True

    return False


def clean_page_text(raw_text: str) -> str:
    """Apply conservative PDF text cleanup without rewriting legal wording."""
    text = (raw_text or "").replace("\x00", " ")
    lines = [line for line in text.splitlines() if not is_header_footer_line(line)]

    # Flatten page text and normalize spacing.
    text = " ".join(lines)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


sample = "Artificial Intelligence Act\nOJ L, 12.7.2024\nELI: http://data.europa.eu/eli/reg/2024/1689/oj"
clean_page_text(sample)

In [ ]:
doc = pymupdf.open(str(PDF_PATH))
records = []

for page_index, page in enumerate(doc, start=1):
    raw_text = page.get_text("text") or ""
    clean_text = clean_page_text(raw_text)
    records.append(
        {
            "source": PDF_PATH.name,
            "page": page_index,
            "raw_text": raw_text,
            "clean_text": clean_text,
            "raw_char_count": len(raw_text),
            "clean_char_count": len(clean_text),
        }
    )

pages_df = pd.DataFrame(records)
pages_df.to_csv(PAGES_CSV, index=False, encoding="utf-8")

print(f"pages extracted: {len(pages_df)}")
print(f"saved to: {PAGES_CSV}")
pages_df[["page", "raw_char_count", "clean_char_count"]].describe()

In [ ]:
# Inspect representative pages before moving to chunking.
for page_number in [1, 44, 53, 100, 124, 144]:
    row = pages_df.loc[pages_df["page"] == page_number].iloc[0]
    print(f"\n--- PAGE {page_number} ---")
    print(row["clean_text"][:1000])

### Step 1 Completion Criteria

Step 1 is complete when:

- `outputs/pages.csv` exists,
- it has one row per PDF page,
- page numbers are preserved,
- `raw_text` and `clean_text` are both available,
- sampled pages are readable enough to proceed to chunking.

## Step 2: Page-Aware Continuous Chunking

Goal for this step:

- Use `clean_text` from `outputs/pages.csv`.
- Treat pages as metadata, not hard chunk boundaries.
- Build a continuous word stream across the full document.
- Prefer legal structure boundaries such as chapters, sections, articles, annexes, clauses, and list items.
- Enforce bounded chunk sizes with overlap.
- Save retrieval-ready chunks to `outputs/chunks.csv`.

In [ ]:
MIN_CHUNK_WORDS = 250
TARGET_CHUNK_WORDS = 600
MAX_CHUNK_WORDS = 900
OVERLAP_WORDS = 100

MIN_CHUNK_WORDS, TARGET_CHUNK_WORDS, MAX_CHUNK_WORDS, OVERLAP_WORDS

In [ ]:
def build_word_stream(pages: pd.DataFrame) -> list[dict]:
    """Create an ordered word stream while retaining the source page for each word."""
    stream = []
    for row in pages.sort_values("page").itertuples(index=False):
        for word in str(row.clean_text).split():
            stream.append({"word": word, "page": int(row.page)})
    return stream


def is_roman_token(token: str) -> bool:
    return bool(re.fullmatch(r"[IVXLCDM]+", token.strip(".:-")))


def is_structural_boundary(words: list[str], index: int) -> bool:
    """Detect likely legal structure starts in a token stream."""
    token = words[index]
    next_token = words[index + 1] if index + 1 < len(words) else ""

    if token in {"CHAPTER", "SECTION", "ANNEX"}:
        return True
    if token == "Article" and re.fullmatch(r"\d+[A-Z]?", next_token.strip(".,;:")):
        return True
    if re.fullmatch(r"\(\d+\)", token):
        return True
    if re.fullmatch(r"\([a-z]\)", token):
        return True
    if re.fullmatch(r"\([ivxlcdm]+\)", token, flags=re.IGNORECASE):
        return True
    if re.fullmatch(r"\d+\.", token) and next_token[:1].isupper():
        return True

    return False


def split_into_structural_units(stream: list[dict]) -> list[tuple[int, int]]:
    words = [item["word"] for item in stream]
    boundaries = [0]

    for index in range(1, len(words)):
        if is_structural_boundary(words, index):
            boundaries.append(index)

    boundaries = sorted(set(boundaries + [len(words)]))
    return [(boundaries[i], boundaries[i + 1]) for i in range(len(boundaries) - 1)]


def structure_hint(text: str) -> str:
    words = text.split()
    return " ".join(words[:16])


def make_chunk_record(chunk_number: int, stream: list[dict], start: int, end: int, overlap_start: int) -> dict:
    items = stream[overlap_start:end]
    text = " ".join(item["word"] for item in items)
    pages = [item["page"] for item in items]

    return {
        "chunk_id": f"C{chunk_number:04d}",
        "text": text,
        "page_start": min(pages),
        "page_end": max(pages),
        "word_count": len(items),
        "char_count": len(text),
        "start_word": overlap_start,
        "end_word": end,
        "core_start_word": start,
        "overlap_words": start - overlap_start,
        "structure_hint": structure_hint(text),
    }


def build_chunks(stream: list[dict], units: list[tuple[int, int]]) -> pd.DataFrame:
    core_spans = []
    current_start = None
    current_end = None

    def flush_current():
        nonlocal current_start, current_end
        if current_start is not None and current_end is not None and current_end > current_start:
            core_spans.append((current_start, current_end))
        current_start = None
        current_end = None

    for unit_start, unit_end in units:
        unit_len = unit_end - unit_start

        if unit_len > MAX_CHUNK_WORDS:
            flush_current()
            split_start = unit_start
            while split_start < unit_end:
                split_end = min(split_start + TARGET_CHUNK_WORDS, unit_end)
                core_spans.append((split_start, split_end))
                split_start = split_end
            continue

        if current_start is None:
            current_start = unit_start
            current_end = unit_end
            continue

        current_len = current_end - current_start
        proposed_len = unit_end - current_start

        if proposed_len <= TARGET_CHUNK_WORDS or (current_len < MIN_CHUNK_WORDS and proposed_len <= MAX_CHUNK_WORDS):
            current_end = unit_end
        else:
            flush_current()
            current_start = unit_start
            current_end = unit_end

    flush_current()

    if len(core_spans) >= 2 and core_spans[-1][1] - core_spans[-1][0] < MIN_CHUNK_WORDS:
        previous_start, _ = core_spans[-2]
        _, last_end = core_spans[-1]
        core_spans[-2:] = [(previous_start, last_end)]

    records = []
    for chunk_index, (start, end) in enumerate(core_spans, start=1):
        overlap_start = start if chunk_index == 1 else max(0, start - OVERLAP_WORDS)
        records.append(make_chunk_record(chunk_index, stream, start, end, overlap_start))

    return pd.DataFrame(records)

In [ ]:
pages_for_chunking = pd.read_csv(PAGES_CSV, encoding="utf-8")
word_stream = build_word_stream(pages_for_chunking)
structural_units = split_into_structural_units(word_stream)
chunks_df = build_chunks(word_stream, structural_units)
chunks_df.to_csv(CHUNKS_CSV, index=False, encoding="utf-8")

print(f"word stream length: {len(word_stream):,}")
print(f"structural units: {len(structural_units):,}")
print(f"chunks created: {len(chunks_df):,}")
print(f"saved to: {CHUNKS_CSV}")
chunks_df[["word_count", "char_count", "overlap_words"]].describe()

In [ ]:
# Inspect several chunks, including page-spanning chunks.
print(chunks_df[["chunk_id", "page_start", "page_end", "word_count", "overlap_words", "structure_hint"]].head(10).to_string(index=False))

page_spanning = chunks_df[chunks_df["page_start"] != chunks_df["page_end"]]
print(f"\npage-spanning chunks: {len(page_spanning)}")

for row in chunks_df.iloc[[0, min(5, len(chunks_df)-1), len(chunks_df)-1]].itertuples(index=False):
    print(f"\n--- {row.chunk_id} | pages {row.page_start}-{row.page_end} | words {row.word_count} ---")
    print(row.text[:1200])

## Step 3: Baseline Dense Retrieval

Goal for this step:

- Use a bi-encoder to embed every chunk independently.
- Encode each query with the same model.
- Rank chunks by cosine similarity.
- Retrieve the top 10 chunks.
- Use the top 3 chunks as the baseline context in the next generation step.

For this small corpus, NumPy cosine similarity is enough. A vector database or FAISS index is not necessary.

In [ ]:
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"

EMBEDDING_MODEL_NAME = str(PROJECT_ROOT / "models" / "all-MiniLM-L6-v2")
TOP_K_RETRIEVAL = 10
TOP_K_CONTEXT = 3

EMBEDDING_MODEL_NAME, TOP_K_RETRIEVAL, TOP_K_CONTEXT

In [ ]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
embedding_model

In [ ]:
chunks_for_retrieval = pd.read_csv(CHUNKS_CSV, encoding="utf-8")
chunk_texts = chunks_for_retrieval["text"].astype(str).tolist()

start_time = time.perf_counter()
chunk_embeddings = embedding_model.encode(
    chunk_texts,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
embedding_seconds = time.perf_counter() - start_time

np.save(EMBEDDINGS_PATH, chunk_embeddings)

print(f"chunks embedded: {len(chunk_embeddings)}")
print(f"embedding dimension: {chunk_embeddings.shape[1]}")
print(f"embedding time: {embedding_seconds:.2f} seconds")
print(f"saved to: {EMBEDDINGS_PATH}")

In [ ]:
def baseline_retrieve(query: str, top_k: int = TOP_K_RETRIEVAL) -> pd.DataFrame:
    """Retrieve top-k chunks with bi-encoder cosine similarity."""
    start_time = time.perf_counter()
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
    )[0]
    scores = chunk_embeddings @ query_embedding
    top_indices = np.argsort(-scores)[:top_k]
    retrieval_seconds = time.perf_counter() - start_time

    results = chunks_for_retrieval.iloc[top_indices].copy()
    results.insert(0, "query", query)
    results.insert(1, "baseline_rank", range(1, len(results) + 1))
    results.insert(2, "bi_encoder_score", scores[top_indices])
    results["baseline_retrieval_seconds"] = retrieval_seconds
    return results


def display_retrieval_results(results: pd.DataFrame, preview_chars: int = 350) -> None:
    for row in results.itertuples(index=False):
        print(f"#{row.baseline_rank} | {row.chunk_id} | pages {row.page_start}-{row.page_end} | score {row.bi_encoder_score:.4f}")
        print(row.text[:preview_chars].replace("\n", " "))
        print()

In [ ]:
candidate_queries = [
    "Which AI practices are prohibited because they manipulate a person's behaviour?",
    "When can law enforcement use real-time remote biometric identification in public spaces?",
    "Which AI systems used in education and vocational training are high-risk?",
    "What transparency obligations apply when users interact with an AI system?",
    "What disclosure is required for AI-generated deep fake content?",
    "What obligations apply to providers of general-purpose AI models with systemic risk?",
    "What criteria are used to designate general-purpose AI models with systemic risk?",
    "What penalties apply for non-compliance with prohibited AI practices?",
]

all_baseline_results = []
for query in candidate_queries:
    all_baseline_results.append(baseline_retrieve(query, top_k=TOP_K_RETRIEVAL))

baseline_results_df = pd.concat(all_baseline_results, ignore_index=True)
baseline_results_df.to_csv(BASELINE_RESULTS_CSV, index=False, encoding="utf-8")

print(f"saved baseline results to: {BASELINE_RESULTS_CSV}")
baseline_results_df[["query", "baseline_rank", "chunk_id", "page_start", "page_end", "bi_encoder_score"]].head(20)

In [ ]:
query_to_inspect = candidate_queries[1]
print(query_to_inspect)
print()
display_retrieval_results(baseline_retrieve(query_to_inspect, top_k=TOP_K_RETRIEVAL))

## Step 4: Cross-Encoder Re-Ranking

Goal for this step:

- Take the top 10 chunks from baseline dense retrieval.
- Score each `(query, chunk)` pair with a cross-encoder.
- Sort the 10 candidates by cross-encoder score.
- Select the top 3 re-ranked chunks for the advanced RAG pipeline.
- Record rank shifts and latency.

In [ ]:
CROSS_ENCODER_MODEL_NAME = str(PROJECT_ROOT / "models" / "ms-marco-MiniLM-L-6-v2")

CROSS_ENCODER_MODEL_NAME

In [ ]:
cross_encoder = CrossEncoder(CROSS_ENCODER_MODEL_NAME)
cross_encoder

In [ ]:
def rerank_baseline_results(query: str, baseline_results: pd.DataFrame) -> tuple[pd.DataFrame, float]:
    """Re-rank baseline top-k results with a cross-encoder."""
    candidates = baseline_results.copy().reset_index(drop=True)
    pairs = [(query, text) for text in candidates["text"].astype(str).tolist()]

    start_time = time.perf_counter()
    cross_scores = cross_encoder.predict(pairs)
    reranking_seconds = time.perf_counter() - start_time

    candidates["cross_encoder_score"] = cross_scores
    candidates = candidates.sort_values("cross_encoder_score", ascending=False).reset_index(drop=True)
    candidates.insert(2, "reranked_rank", range(1, len(candidates) + 1))
    candidates["reranking_seconds"] = reranking_seconds
    candidates["total_reranking_pipeline_seconds"] = candidates["baseline_retrieval_seconds"].iloc[0] + reranking_seconds
    return candidates, reranking_seconds


def display_rank_shift(reranked_results: pd.DataFrame, preview_chars: int = 300) -> None:
    for row in reranked_results.itertuples(index=False):
        print(
            f"rerank #{row.reranked_rank} | baseline #{row.baseline_rank} | {row.chunk_id} | "
            f"pages {row.page_start}-{row.page_end} | bi {row.bi_encoder_score:.4f} | cross {row.cross_encoder_score:.4f}"
        )
        print(row.text[:preview_chars].replace("\n", " "))
        print()

In [ ]:
reranked_frames = []
latency_records = []

for query in candidate_queries:
    baseline_for_query = baseline_results_df[baseline_results_df["query"] == query].copy()
    reranked_for_query, reranking_seconds = rerank_baseline_results(query, baseline_for_query)
    reranked_frames.append(reranked_for_query)

    latency_records.append(
        {
            "query": query,
            "baseline_retrieval_seconds": float(baseline_for_query["baseline_retrieval_seconds"].iloc[0]),
            "reranking_seconds": float(reranking_seconds),
            "total_reranking_pipeline_seconds": float(
                baseline_for_query["baseline_retrieval_seconds"].iloc[0] + reranking_seconds
            ),
        }
    )

reranked_results_df = pd.concat(reranked_frames, ignore_index=True)
latency_results_df = pd.DataFrame(latency_records)

reranked_results_df.to_csv(RERANKED_RESULTS_CSV, index=False, encoding="utf-8")
latency_results_df.to_csv(LATENCY_RESULTS_CSV, index=False, encoding="utf-8")

print(f"saved reranked results to: {RERANKED_RESULTS_CSV}")
print(f"saved latency results to: {LATENCY_RESULTS_CSV}")
latency_results_df.describe()

In [ ]:
query_to_inspect = candidate_queries[1]
print(query_to_inspect)
print()
display_rank_shift(reranked_results_df[reranked_results_df["query"] == query_to_inspect])

In [ ]:
rank_shift_summary = (
    reranked_results_df[reranked_results_df["reranked_rank"] <= TOP_K_CONTEXT]
    [["query", "reranked_rank", "baseline_rank", "chunk_id", "page_start", "page_end", "bi_encoder_score", "cross_encoder_score"]]
    .sort_values(["query", "reranked_rank"])
)

rank_shift_summary

## Step 5: Prompt Assembly for Answer Generation

Goal for this step:

- Select the strongest failure-query candidates.
- Build Answer 1 prompts using the baseline top 3 chunks.
- Build Answer 2 prompts using the re-ranked top 3 chunks.
- Use the exact same instruction template for both pipelines.
- Export prompts to `outputs/llm_prompts.csv`.

This notebook exports prompts so they can be run through the chosen LLM provider. The retrieval comparison remains reproducible even if the LLM API is changed.

In [ ]:
FINAL_FAILURE_QUERIES = [
    "What obligations apply to providers of general-purpose AI models with systemic risk?",
    "When can law enforcement use real-time remote biometric identification in public spaces?",
]


def format_context(results: pd.DataFrame, rank_column: str, top_k: int = TOP_K_CONTEXT) -> str:
    selected = results.sort_values(rank_column).head(top_k).reset_index(drop=True)
    passages = []
    for index, row in enumerate(selected.itertuples(index=False), start=1):
        passages.append(
            f"[S{index}] Chunk {row.chunk_id}, pages {row.page_start}-{row.page_end}:\n{row.text}"
        )
    return "\n\n".join(passages)


def build_prompt(query: str, context: str) -> str:
    return f"""Answer using only the numbered passages below.
If the passages do not contain the answer, say that the document does not provide enough information.
Cite every factual claim using passage ids like [S1], [S2].

Context:
{context}

Question: {query}
""".strip()


def get_baseline_top3(query: str) -> pd.DataFrame:
    return baseline_results_df[baseline_results_df["query"] == query].sort_values("baseline_rank").head(TOP_K_CONTEXT)


def get_reranked_top3(query: str) -> pd.DataFrame:
    return reranked_results_df[reranked_results_df["query"] == query].sort_values("reranked_rank").head(TOP_K_CONTEXT)

In [ ]:
prompt_records = []

for query in FINAL_FAILURE_QUERIES:
    baseline_context = format_context(get_baseline_top3(query), rank_column="baseline_rank")
    reranked_context = format_context(get_reranked_top3(query), rank_column="reranked_rank")

    prompt_records.append(
        {
            "query": query,
            "pipeline": "baseline",
            "context_chunk_ids": ", ".join(get_baseline_top3(query)["chunk_id"].tolist()),
            "prompt": build_prompt(query, baseline_context),
        }
    )
    prompt_records.append(
        {
            "query": query,
            "pipeline": "reranked",
            "context_chunk_ids": ", ".join(get_reranked_top3(query)["chunk_id"].tolist()),
            "prompt": build_prompt(query, reranked_context),
        }
    )

llm_prompts_df = pd.DataFrame(prompt_records)
llm_prompts_df.to_csv(LLM_PROMPTS_CSV, index=False, encoding="utf-8")

print(f"saved prompts to: {LLM_PROMPTS_CSV}")
llm_prompts_df[["query", "pipeline", "context_chunk_ids"]]

In [ ]:
print(llm_prompts_df.iloc[0]["prompt"][:2500])

## Step 6: Answer Generation

The prompts for the selected failure queries have been exported to `outputs/llm_prompts.csv`.

Run each prompt through the same LLM with the same generation settings. Save the generated answers to `outputs/llm_answers.csv` using the schema below.

Do not store API keys or provider credentials in this notebook or repository.

In [ ]:
answer_template_df = llm_prompts_df[["query", "pipeline", "context_chunk_ids"]].copy()
answer_template_df["model"] = "TODO"
answer_template_df["generation_seconds"] = "TODO"
answer_template_df["answer"] = "TODO"

if not LLM_ANSWERS_CSV.exists():
    answer_template_df.to_csv(LLM_ANSWERS_CSV, index=False, encoding="utf-8")
    print(f"created answer template: {LLM_ANSWERS_CSV}")
else:
    print(f"answer file already exists: {LLM_ANSWERS_CSV}")

answer_template_df